# Parallel repeat decisions

This notebook replays the exact retest-plan arithmetic and reads the retained aggregate experiment. It does not load restricted player data or turn a retained aggregate into an independent real-data rerun. The full study, assumptions and source references are in `docs/nfl/PARALLEL_REPEAT_DECISIONS.md`.

The complete companion ZIP includes all per-refit aggregate rows and complete test receipts in `run.json.gz`; the GitHub source snapshot also provides a hash-bound `PUBLIC_EVIDENCE.json` summary. The notebook supports either representation and labels the same retained development evidence.

In [1]:
from pathlib import Path
import sys, gzip, json, hashlib
root = next(p for p in (Path.cwd(), Path.cwd().parent, Path.cwd()/"PerfectJet") if (p/"src/perfectjet").is_dir())
sys.path.insert(0, str(root/"src"))
folder = root/"results/parallel_repeat_decisions"
run = json.loads(gzip.decompress((folder/"run.json.gz").read_bytes())) if (folder/"run.json.gz").exists() else json.loads((folder/"PUBLIC_EVIDENCE.json").read_text())
for relative, expected in run["execution"]["source_sha256"].items():
    assert hashlib.sha256((root/relative).read_bytes()).hexdigest() == expected, relative
print("Source hashes matched. Original fit/calibration/evaluation players:", *[run["study"][k] for k in ("fit_players","calibration_players","evaluation_players")])

Source hashes matched. Original fit/calibration/evaluation players: 29 26 29


## Exact population arithmetic

The following is a constructed scalar model, not football evidence. It demonstrates exact feasible-plan counting, inverse ranking and asymmetric versus equal repetition costs.

In [2]:
from perfectjet.repeat_allocation import RepeatPlanPopulation
from perfectjet.replication_geometry import scalar_repeat_budget, calibration_requirement
p = RepeatPlanPopulation.build(1,1,1,1,"2.3",1,budget=10**30)
assert p.count() == (10**30+1)*(10**30+2)//2-10
assert p.rank(p.unrank(p.count()-1)) == p.count()-1
print("Exact feasible plans:", p.count())
print("Minimum additional cost and total-attempt allocations:", p.minimum())
print("Equal-repetition additional cost:", 2*(scalar_repeat_budget(1,1,"2.3",1)["repeats"]-1))
print("Finite-radius calibration requirement for 29-player, 90% Bonferroni board:", calibration_requirement("0.1",29))

Exact feasible plans: 500000000000000000000000000001499999999999999999999999999991
Minimum additional cost and total-attempt allocations: {'additional_cost': 3, 'plans': [(2, 3), (3, 2)]}
Equal-repetition additional cost: 4
Finite-radius calibration requirement for 29-player, 90% Bonferroni board: 289


## Retained real-data comparison

These are measurements from the same 29 evaluation athletes. The three ridge rows share identical predictions and differ only in their uncertainty geometry. Region coverage is descriptive for the 2025 development class, not a new guarantee of coverage under class shift.

In [3]:
import pandas as pd
rows=[]
for name,value in run["study"]["results"].items():
    rows.append({"method":name,"MSE":value["mse"],"joint_covered":value["joint_calibrated_covered"],"players":29,"volume":value["ellipsoid_volume"],"resolved_mean_score_pairs":value["marginal_card_comparisons"]["settled_pairs"]})
print(pd.DataFrame(rows).to_string(index=False))
print("Completed original-player sensitivity refits:",run["refits"]["successful_draws"])
print("Unique completed test cases:",run["tests"]["unique_passed"])

         method      MSE  joint_covered  players    volume  resolved_mean_score_pairs
            raw 0.143948             29       29 21.235912                          0
       reliable 0.104558             29       29 15.866623                          0
 ridge_diagonal 0.100159             29       29 12.567157                          0
ridge_isotropic 0.100159             29       29  9.502854                          0
    ridge_joint 0.100159             29       29 14.668055                          0
Completed original-player sensitivity refits: 128
Unique completed test cases: 377


## Interpretation

Ordinary ridge beats raw reuse of the first attempt on this repeated-measurement target. Full covariance does not beat the isotropic uncertainty control in the retained split. The joint prediction regions contain the measured evaluation vectors but do not separate any pair on the declared mean score. The exact allocation engine can be used conditionally on a specified planning model; its integer answers do not establish football transfer, real retest savings or an NFL scouting edge. Complete owner-data reruns use `scripts/run_repeat_decisions.py` with the separately supplied trusted private checkpoint.